# Semana 03: Máquinas Virtuais (AWS EC2) e Conteinerização com Docker

## Projetos 01 & 04 — Fábrica Virtual Smart N1

Este notebook apresenta os fundamentos de **Amazon EC2** e **Docker**, seguidos de uma atividade prática no **AWS Learner Lab** para provisionar uma máquina virtual Linux e executar uma aplicação web em um container.

### Objetivos de aprendizagem
- Compreender AMI, tipos de instância, EBS e Security Groups.
- Entender acesso remoto por SSH e EC2 Instance Connect.
- Diferenciar máquinas virtuais e containers.
- Provisionar uma instância Ubuntu no AWS Learner Lab.
- Instalar o Docker Engine.
- Executar um servidor Nginx em container.
- Publicar uma página web da Fábrica Virtual Smart N1 na porta 80.

---


## 1. Fundamentação teórica

### 1.1 Amazon EC2

O **Amazon EC2 (Elastic Compute Cloud)** é um serviço de computação da AWS que permite criar e executar máquinas virtuais na nuvem.

Os principais componentes utilizados nesta atividade são:

1. **AMI (Amazon Machine Image):** imagem utilizada como modelo para criar a máquina virtual, incluindo o sistema operacional.
2. **Instance Type:** define recursos como CPU, memória e capacidade de rede. No Learner Lab, a disponibilidade depende do laboratório; `t3.micro` ou `t2.micro` são exemplos comuns.
3. **EBS (Elastic Block Store):** armazenamento persistente em blocos associado à instância.
4. **Security Group:** firewall virtual com regras de entrada e saída. É **stateful**, portanto o tráfego de resposta de uma conexão permitida é automaticamente acompanhado.

![Diagrama de Arquitetura AWS EC2 e Docker](images/aws_ec2_docker_arquitetura.jpg)


### 1.2 Portas e conectividade

Nesta atividade serão utilizadas duas portas:

| Porta | Protocolo | Finalidade |
|---:|---|---|
| 22 | TCP/SSH | Acesso administrativo à EC2 |
| 80 | TCP/HTTP | Acesso à aplicação web |

> **Atenção:** liberar SSH (`22`) para `0.0.0.0/0` facilita o laboratório, mas não é uma boa prática para ambientes reais. Sempre que possível, restrinja a origem ao seu IP ou utilize mecanismos de acesso mais seguros.


### 1.3 Máquinas virtuais × containers

![Diagrama Comparativo de Arquitetura VMs vs Containers](images/vms_vs_containers_docker.jpg)

**Máquinas virtuais** executam um sistema operacional convidado completo sobre uma camada de virtualização (hypervisor). Cada VM possui seu próprio kernel.

**Containers** isolam processos e compartilham o kernel do sistema operacional hospedeiro. Por isso, normalmente apresentam menor sobrecarga e inicialização mais rápida.

No Docker:

- **Docker Engine:** plataforma responsável por executar e gerenciar containers.
- **Imagem:** pacote somente leitura contendo aplicação, dependências e arquivos necessários à execução.
- **Container:** instância em execução de uma imagem.
- **Port mapping:** `-p 80:80` publica a porta 80 do host e a direciona para a porta 80 do container.

---


## 2. Arquitetura da atividade

O fluxo será:

**Internet → Security Group → EC2 Ubuntu → Docker Engine → Container Nginx → Página Smart N1**

A porta 80 precisa estar liberada no Security Group e publicada pelo Docker para que a aplicação possa ser acessada externamente.

---


## 3. Prática — AWS Learner Lab

### Passo 1 — Criar o Security Group

No console da AWS:

1. Acesse **EC2 → Security Groups**.
2. Clique em **Create security group**.
3. Nome: `sg-smartn1-web`.
4. Descrição: `Libera SSH e HTTP para a Fabrica Smart N1`.
5. Selecione a VPC utilizada pelo laboratório.
6. Em **Inbound rules**, adicione:
   - **SSH / TCP / 22** — origem adequada ao laboratório.
   - **HTTP / TCP / 80** — `0.0.0.0/0` para permitir acesso web público.
7. Mantenha as regras de saída padrão, salvo orientação diferente do laboratório.


### Passo 2 — Criar a instância EC2

No painel **EC2**, selecione **Launch instance**.

Configuração sugerida:

- **Name:** `ec2-smartn1-web`
- **AMI:** Ubuntu Server 22.04 LTS (64-bit x86), se disponível no laboratório
- **Instance type:** `t3.micro` ou outro tipo permitido pelo Learner Lab
- **Key pair:** `vockey`, quando disponibilizado pelo laboratório
- **Network:** selecione o Security Group `sg-smartn1-web`
- **IAM Instance Profile:** `LabInstanceProfile` ou `LabRole`, caso esteja disponível

Clique em **Launch instance** e aguarde o estado **Running**.


### Passo 3 — Conectar à EC2

A forma mais simples no Learner Lab é utilizar **EC2 Instance Connect**:

1. Selecione `ec2-smartn1-web`.
2. Clique em **Connect**.
3. Selecione **EC2 Instance Connect**.
4. Clique em **Connect**.

Alternativamente, em um terminal local com a chave privada adequada:

```bash
ssh -i "vockey.pem" ubuntu@IP_PUBLICO_DA_EC2
```

> Os comandos das próximas células devem ser executados **no terminal da EC2**. O prefixo `!`, usado em notebooks Jupyter para executar comandos do sistema, não é necessário no terminal SSH.


### Passo 4 — Preparar o sistema operacional

Execute no terminal da EC2:

In [ ]:
sudo apt-get update -y
sudo apt-get upgrade -y


### Passo 5 — Instalar o Docker Engine

Para fins didáticos, o procedimento abaixo utiliza o script oficial de instalação do Docker.

> Em ambientes de produção, recomenda-se avaliar cuidadosamente o método de instalação e utilizar os repositórios oficiais apropriados à distribuição Linux.

In [ ]:
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh


In [ ]:
sudo systemctl enable --now docker
sudo usermod -aG docker "$USER"
newgrp docker


### Passo 6 — Executar o servidor web Nginx

O comando abaixo cria um container chamado `hello-web-smartn1` a partir da imagem `nginx:alpine`.

- `-d`: executa em segundo plano (*detached mode*).
- `--name`: define o nome do container.
- `--restart always`: solicita reinício automático do container em situações suportadas pelo Docker.
- `-p 80:80`: publica a porta 80 da EC2 na porta 80 do container.


In [ ]:
docker run -d --name hello-web-smartn1 --restart always -p 80:80 nginx:alpine


### Passo 7 — Criar a página da Fábrica Virtual Smart N1

Criaremos uma página HTML diretamente na EC2 e depois a copiaremos para o diretório servido pelo Nginx.

In [ ]:
cat > index.html <<'EOF'
<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Smart N1 - Indústria 4.0 Cloud</title>
</head>
<body style="font-family:Arial,sans-serif;background:#0d1117;color:#58a6ff;text-align:center;padding-top:50px">
  <h1>Fábrica Virtual Smart N1 - AWS Cloud</h1>
  <h2>Projeto 01 e 04: Servidor Web em Container Docker na EC2</h2>
  <p>Status da Linha de Produção: <strong>OPERACIONAL</strong></p>
  <p>Aplicação executada com Docker + Nginx.</p>
</body>
</html>
EOF
cat index.html


### Passo 8 — Publicar o HTML no container e validar


In [ ]:
docker cp index.html hello-web-smartn1:/usr/share/nginx/html/index.html
curl -I http://localhost:80


---

## 4. Exercícios de fixação e avaliação

### Questão 1
Qual é a função das regras de entrada (*Inbound Rules*) de um Security Group? O que aconteceria com o acesso externo à aplicação se a porta 80 não estivesse liberada?

### Questão 2
Explique a função dos parâmetros `-d`, `--restart always` e `-p 80:80` utilizados no comando `docker run`.

### Questão 3
Explique por que containers normalmente apresentam menor sobrecarga que máquinas virtuais tradicionais.

### Questão 4
Descreva o caminho percorrido por uma requisição HTTP feita ao endereço público da EC2 até chegar ao Nginx dentro do container.

### Questão 5 — Evidência prática
Apresente uma captura de tela contendo:

- A instância EC2 em estado **Running**;
- O endereço IPv4 público;
- O comando `docker ps` mostrando o container em execução;
- A página Smart N1 aberta no navegador.
